# 00 — Normalização e Formatação da Mega-Sena Geral

> **Projeto:** LSTM para a Mega da Virada  
> **Arquivo alvo:** `data/raw/megasena-download-resultados.csv`  
> **Requisitos associados:** RT-001, RT-003, RT-004

---

### Contexto e Objetivo
O arquivo original baixado da Caixa possui uma divergência estrutural entre o cabeçalho e os dados:
- Cabeçalho: `Concurso,Data,Números` (3 campos)
- Linhas de dados: `1,10/03/1996,4,5,30,33,41,52` (8 campos separados por vírgula)

Este notebook padroniza o histórico oficial para o mesmo formato tabular de `mega_da_virada.csv`:
```text
Ano,Concurso,Data,N1,N2,N3,N4,N5,N6
```
*(ou sem `Data`, se desejado: `Ano,Concurso,N1,N2,N3,N4,N5,N6`)*

### Regras de transformação:
1. Extração do campo **Ano** a partir da data de realização do sorteio.
2. Ordenação crescente estrita das 6 dezenas sorteadas: $N1 < N2 < N3 < N4 < N5 < N6$.
3. Formatação das dezenas com dois dígitos inteiros (`01` a `60`).
4. Validação completa de integridade (RT-003): sem lacunas, intervalo $[1, 60]$, sem duplicatas.
5. Gravação segura com substituição atômica protegida pela trava `CONFIRMAR_ALTERACAO = False`.

## 1. Configuração de Caminhos e Hash Inicial

In [1]:
import os
import shutil
import hashlib
from pathlib import Path
import pandas as pd
import numpy as np

DATA_RAW_DIR = Path("../data/raw")
MEGASENA_CSV_PATH = DATA_RAW_DIR / "megasena-download-resultados.csv"

def compute_sha256(filepath: Path) -> str:
    """Calcula o hash SHA-256 de um arquivo em chunks."""
    sha256 = hashlib.sha256()
    with open(filepath, "rb") as f:
        while chunk := f.read(65536):
            sha256.update(chunk)
    return sha256.hexdigest()

sha256_inicial = compute_sha256(MEGASENA_CSV_PATH)
print(f"Arquivo alvo: {MEGASENA_CSV_PATH.resolve()}")
print(f"Hash SHA-256 inicial: {sha256_inicial}")


Arquivo alvo: C:\Users\Matheus Pinheiro\OneDrive - Am Tech\Área de Trabalho\Mega da virada LSTM\data\raw\megasena-download-resultados.csv
Hash SHA-256 inicial: 088db01fc12dac505722f37ac04b35506686824c3c12a80582146f5e7afad06c


## 2. Inspeção do Formato Bruto das Linhas

In [2]:
# Inspeção das primeiras 5 linhas brutas em texto
with open(MEGASENA_CSV_PATH, "r", encoding="utf-8-sig", errors="replace") as f:
    for i in range(6):
        linha = f.readline().strip()
        print(f"Linha {i}: {linha}")


Linha 0: Concurso,Data,Números
Linha 1: 1,10/03/1996,4,5,30,33,41,52
Linha 2: 2,17/03/1996,9,37,39,41,43,49
Linha 3: 3,24/03/1996,10,11,29,30,36,47
Linha 4: 4,31/03/1996,1,5,6,27,42,59
Linha 5: 5,07/04/1996,1,2,6,16,19,46


## 3. Leitura e Parsing Estruturado com Pandas

In [3]:
# Como o header original possui 3 campos e os dados têm 8,
# pulamos a linha 0 e atribuímos os nomes explicitamente:
df_raw = pd.read_csv(
    MEGASENA_CSV_PATH,
    skiprows=1,
    header=None,
    names=["Concurso", "Data", "D1", "D2", "D3", "D4", "D5", "D6"],
    encoding="utf-8-sig",
    dtype={
        "Concurso": int,
        "Data": str,
        "D1": int,
        "D2": int,
        "D3": int,
        "D4": int,
        "D5": int,
        "D6": int
    }
)

print(f"Total de sorteios lidos: {len(df_raw)}")
print(f"Colunas brutas: {df_raw.columns.tolist()}")
df_raw.head()


Total de sorteios lidos: 3074
Colunas brutas: ['Concurso', 'Data', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6']


,Concurso,Data,D1,D2,D3,D4,D5,D6
0,1,10/03/1996,4,5,30,33,41,52
1,2,17/03/1996,9,37,39,41,43,49
2,3,24/03/1996,10,11,29,30,36,47
3,4,31/03/1996,1,5,6,27,42,59
4,5,07/04/1996,1,2,6,16,19,46


## 4. Padronização: Ano, Concurso, Data e Dezenas Ordenadas (N1..N6)

- Extração do Ano da Data (`dd/mm/aaaa`).
- Ordenação crescente de cada sorteio ($N1 < N2 < N3 < N4 < N5 < N6$).
- Formatação das dezenas com dois dígitos (`01`, `02`, ..., `60`).

In [4]:
# Parâmetro: manter ou não a coluna Data (por padrão True, para atender RT-003)
INCLUIR_DATA = True

# 1. Extração do Ano
anos = pd.to_datetime(df_raw["Data"], format="%d/%m/%Y").dt.year

# 2. Extração e ordenação das dezenas linha a linha
dezenas_matriz = df_raw[["D1", "D2", "D3", "D4", "D5", "D6"]].values
dezenas_ordenadas = np.sort(dezenas_matriz, axis=1)

# 3. Formatação das dezenas com 2 dígitos (string padronizada com zero à esquerda)
cols_dezenas = [f"N{i}" for i in range(1, 7)]
df_dezenas_fmt = pd.DataFrame(
    [[f"{num:02d}" for num in linha] for linha in dezenas_ordenadas],
    columns=cols_dezenas
)

# 4. Montagem do DataFrame formatado final
if INCLUIR_DATA:
    df_formatado = pd.DataFrame({
        "Ano": anos,
        "Concurso": df_raw["Concurso"],
        "Data": df_raw["Data"]
    }).join(df_dezenas_fmt)
else:
    df_formatado = pd.DataFrame({
        "Ano": anos,
        "Concurso": df_raw["Concurso"]
    }).join(df_dezenas_fmt)

print("Estrutura do DataFrame formatado:")
print(f"Dimensões: {df_formatado.shape}")
print(f"Colunas: {df_formatado.columns.tolist()}")


Estrutura do DataFrame formatado:
Dimensões: (3074, 9)
Colunas: ['Ano', 'Concurso', 'Data', 'N1', 'N2', 'N3', 'N4', 'N5', 'N6']


## 5. Validação de Qualidade e Integridade (RT-003)

Validações automáticas de integridade:
- Quantidade esperada de dezenas por concurso = 6.
- Intervalo das dezenas em $[1, 60]$.
- Dezenas estritamente crescentes ($N1 < N2 < N3 < N4 < N5 < N6$) e sem repetições.
- Concursos estritamente contínuos e sem lacunas (1 a $N$).
- Ausência de valores nulos.

In [5]:
# 1. Checagem de nulos
assert df_formatado.isnull().sum().sum() == 0, "Há valores nulos no dataset!"

# 2. Checagem da continuidade dos concursos
concursos = df_formatado["Concurso"].values
esperados = np.arange(1, len(df_formatado) + 1)
assert np.array_equal(concursos, esperados), "Há lacunas ou duplicatas nos números de concurso!"

# 3. Checagem do intervalo e ordenação das dezenas
dezenas_int = df_formatado[cols_dezenas].astype(int).values
assert (dezenas_int >= 1).all() and (dezenas_int <= 60).all(), "Dezenas fora do intervalo [1, 60]!"
assert (np.diff(dezenas_int, axis=1) > 0).all(), "Dezenas não estão estritamente crescentes ou têm duplicatas!"

print("✅ TODAS AS VALIDAÇÕES DE INTEGRIDADE (RT-003) PASSARAM COM SUCESSO!")
print(f"Total de concursos validados: {len(df_formatado)} (Concursos 1 a {len(df_formatado)})")


✅ TODAS AS VALIDAÇÕES DE INTEGRIDADE (RT-003) PASSARAM COM SUCESSO!
Total de concursos validados: 3074 (Concursos 1 a 3074)


## 6. Prévia do Resultado Final

In [6]:
print("Primeiros 5 concursos:")
print(df_formatado.head())

print("\nÚltimos 5 concursos:")
print(df_formatado.tail())


Primeiros 5 concursos:


,Ano,Concurso,Data,N1,N2,N3,N4,N5,N6
0,1996,1,10/03/1996,04,05,30,33,41,52
1,1996,2,17/03/1996,09,37,39,41,43,49
2,1996,3,24/03/1996,10,11,29,30,36,47
3,1996,4,31/03/1996,01,05,06,27,42,59
4,1996,5,07/04/1996,01,02,06,16,19,46



Últimos 5 concursos:


,Ano,Concurso,Data,N1,N2,N3,N4,N5,N6
3069,2026,3070,14/10/2026,19,27,29,43,54,60
3070,2026,3071,18/10/2026,12,22,41,45,51,52
3071,2026,3072,20/10/2026,02,06,09,16,31,40
3072,2026,3073,22/10/2026,04,13,23,28,41,58
3073,2026,3074,24/10/2026,02,03,20,25,39,57


## 7. Gravação Segura Atômica no Arquivo (PROTEGIDA POR TRAVA)

> **⚠️ TRAVA DE SEGURANÇA:**
> A variável `CONFIRMAR_ALTERACAO` abaixo está intencionalmente definida como `False`.
> Altere para `True` somente quando desejar persistir a alteração no arquivo em disco.

In [9]:
CONFIRMAR_ALTERACAO = False

if not CONFIRMAR_ALTERACAO:
    print("⛔ GRAVAÇÃO BLOQUEADA: CONFIRMAR_ALTERACAO está definido como False.")
    print("Nenhuma alteração foi realizada em disco.")
    print("Para aplicar a formatação no arquivo, mude a variável para True e reexecute esta célula.")
else:
    print("Iniciando gravação atômica segura...")
    temp_file = MEGASENA_CSV_PATH.with_suffix(".tmp")
    
    # Grava no arquivo temporário
    df_formatado.to_csv(temp_file, index=False, encoding="utf-8")
    
    # Substituição atômica no sistema de arquivos
    os.replace(temp_file, MEGASENA_CSV_PATH)
    print("✅ Substituição atômica concluída com sucesso!")


⛔ GRAVAÇÃO BLOQUEADA: CONFIRMAR_ALTERACAO está definido como False.
Nenhuma alteração foi realizada em disco.
Para aplicar a formatação no arquivo, mude a variável para True e reexecute esta célula.


## 8. Verificação Pós-Gravação e Releitura

In [10]:
sha256_final = compute_sha256(MEGASENA_CSV_PATH)
print(f"Hash SHA-256 inicial: {sha256_inicial}")
print(f"Hash SHA-256 atual:   {sha256_final}")
print(f"Arquivo foi alterado: {sha256_inicial != sha256_final}")

# Releitura estrita
df_verificado = pd.read_csv(MEGASENA_CSV_PATH)
print(f"\nTotal de linhas relidas: {len(df_verificado)}")
print("Colunas relidas:", df_verificado.columns.tolist())
df_verificado.head(3)


Hash SHA-256 inicial: 088db01fc12dac505722f37ac04b35506686824c3c12a80582146f5e7afad06c
Hash SHA-256 atual:   043a16a7044d54eed9388629de3205ea2b8340c8f86b14525d3bed99ac3a6cc8
Arquivo foi alterado: True

Total de linhas relidas: 3074
Colunas relidas: ['Ano', 'Concurso', 'Data', 'N1', 'N2', 'N3', 'N4', 'N5', 'N6']


,Ano,Concurso,Data,N1,N2,N3,N4,N5,N6
0,1996,1,10/03/1996,4,5,30,33,41,52
1,1996,2,17/03/1996,9,37,39,41,43,49
2,1996,3,24/03/1996,10,11,29,30,36,47
